In [1]:

%load_ext autoreload
%autoreload 2

import os
import requests
import pandas as pd
from dotenv import load_dotenv
from gasp import features

load_dotenv()
API_KEY = os.getenv("EIA_API_KEY")



In [2]:
url = "https://api.eia.gov/v2/natural-gas/stor/wkly/data/"                                 # TODO: everything before the ? in your browser url

params = {
    "api_key": API_KEY,
    "frequency": "weekly",
    "data[0]": "value",
    "facets[series][]": "NW2_EPG0_SWO_R48_BCF",
    "sort[0][column]": "period",
    "sort[0][direction]": "asc",
    "offset": "0",
    "length": "5000"
}

r = requests.get(url, params=params)                  # TODO: send the GET request with url and params
r.raise_for_status()                                  # TODO: raise an error if the request failed

response = r.json()                          # TODO: turn the response into a python dict
rows = response["response"]["data"]                               # TODO: the list of rows inside response

len(rows)

876

In [3]:

df = pd.DataFrame(rows)[["period", "value"]]

df["period"] = pd.to_datetime(df["period"])
df["value"] = pd.to_numeric(df["value"]).astype(float)
df["series"] = "NW2_EPG0_SWO_R48_BCF"

df = (
    df[["period", "series", "value"]]
    .sort_values("period")
    .reset_index(drop=True)
)

df.head()


,period,series,value
0,2010-01-01,NW2_EPG0_SWO_R48_BCF,3117.0
1,2010-01-08,NW2_EPG0_SWO_R48_BCF,2850.0
2,2010-01-15,NW2_EPG0_SWO_R48_BCF,2607.0
3,2010-01-22,NW2_EPG0_SWO_R48_BCF,2521.0
4,2010-01-29,NW2_EPG0_SWO_R48_BCF,2406.0


In [4]:

result = features.add_calendar(df)
result = features.add_change(result)
result = features.add_lags(result)
result = features.add_five_year_stats(result)

result.tail()


,period,series,value,iso_year,iso_week,weekly_change,lag_1,lag_2,lag_52,five_year_min,five_year_max,five_year_mean
871,2026-09-11,NW2_EPG0_SWO_R48_BCF,3298.0,2026,37,44.0,3254.0,3214.0,3423.0,2874.0,3445.0,3218.6
872,2026-09-18,NW2_EPG0_SWO_R48_BCF,3351.0,2026,38,53.0,3298.0,3254.0,3499.0,2977.0,3499.0,3299.6
873,2026-09-25,NW2_EPG0_SWO_R48_BCF,3415.0,2026,39,64.0,3351.0,3298.0,3551.0,3106.0,3551.0,3387.6
874,2026-10-02,NW2_EPG0_SWO_R48_BCF,3500.0,2026,40,85.0,3415.0,3351.0,3631.0,3231.0,3631.0,3478.0
875,2026-10-09,NW2_EPG0_SWO_R48_BCF,3567.0,2026,41,67.0,3500.0,3415.0,3712.0,3342.0,3712.0,3569.4


In [5]:
check_year = 2026
check_week = 20

historical = result[
    (result["iso_week"] == check_week)
    & (result["iso_year"] >= check_year - 5)
    & (result["iso_year"] < check_year)
]

print(historical[["period", "value"]])
print(historical["value"].mean())

        period   value
594 2021-05-21  2215.0
646 2022-05-20  1819.0
698 2023-05-19  2336.0
750 2024-05-17  2719.0
802 2025-05-16  2375.0
2292.8


In [6]:

from pandas.testing import assert_frame_equal

def build_features(data):
    result = features.add_calendar(data)
    result = features.add_change(result)
    result = features.add_lags(result)
    result = features.add_five_year_stats(result)
    return result

full = build_features(df)

cut = len(df) - 20
partial = build_features(df.iloc[:cut].copy())

assert_frame_equal(
    full.iloc[:cut].reset_index(drop=True),
    partial.reset_index(drop=True)
)

print("Leakage test passed!")


Leakage test passed!
